# Construcción del dataset integrado

**Objetivo:** generar `dataset_integrado.csv` — tabla maestra clima + producción por `(región, cultivo, año, mes)` lista para **clustering** y modelado.

| Campo | Valor |
|-------|-------|
| Input MIDAGRI | `OUTPUTS/midagri_largo.csv` (generado por `01_midagri_pipeline.ipynb`) |
| Input NASA | `OUTPUTS/nasa_2020_2025.csv` (generado por `02_nasa_pipeline.ipynb`) |
| Input mapping | `BDS/mapping_cultivo_distrito.csv` |
| Output maestro | `OUTPUTS/dataset_integrado.csv` |

**Correcciones aplicadas (vs notebook 03 original):**
- Pareto-80: incluye cultivos hasta alcanzar **≥ 80%** de producción regional
- Agregado regional: `sum(min_count=1)` — meses sin dato MIDAGRI quedan como NaN, no como 0

**Decisiones:** NaN explícitos en meses faltantes; ceros estructurales conservados; sin remoción de outliers.

**Nombres de columnas:** variables NASA renombradas a español (`temp_promedio`, `precipitacion`, etc.). Ver `DOCUMENTACIÓN/dataset_integrado.md`.


In [6]:
# §0 Setup
# En Colab: descomentar la siguiente línea
# !pip install pandas numpy openpyxl requests -q

import sys
from pathlib import Path

import pandas as pd

# Rutas del proyecto (ajustar en Colab si es necesario)
ROOT = Path(".").resolve()
if not (ROOT / "OUTPUTS").exists() and (ROOT.parent / "OUTPUTS").exists():
    ROOT = ROOT.parent

SCRIPTS = ROOT / "SCRIPTS"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

RUTA_OUTPUT = ROOT / "OUTPUTS"
RUTA_MAPPING = ROOT / "BDS" / "mapping_cultivo_distrito.csv"
RUTA_BASE = ROOT / "BDS"  # Excel MIDAGRI (solo si se re-ejecuta NB01)

RUTA_OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 25)

print(f"ROOT: {ROOT}")
print(f"OUTPUT: {RUTA_OUTPUT}")


ROOT: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF
OUTPUT: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\OUTPUTS


## §1 Verificar insumos

Ejecuta primero los notebooks `01_midagri_pipeline.ipynb` y `02_nasa_pipeline.ipynb`,  
o coloca los CSV intermedios en `OUTPUTS/`.


In [7]:
insumos = {
    "midagri_largo.csv": RUTA_OUTPUT / "midagri_largo.csv",
    "nasa_2020_2025.csv": RUTA_OUTPUT / "nasa_2020_2025.csv",
    "mapping": RUTA_MAPPING,
}

for nombre, ruta in insumos.items():
    ok = "OK" if ruta.exists() else "FALTA"
    print(f"  [{ok}] {nombre}: {ruta}")

faltantes = [n for n, r in insumos.items() if not r.exists()]
if faltantes:
    raise FileNotFoundError(
        f"Faltan insumos: {faltantes}. Ejecuta notebooks 01 y 02 primero."
    )
print("\nInsumos listos.")


  [OK] midagri_largo.csv: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\OUTPUTS\midagri_largo.csv
  [OK] nasa_2020_2025.csv: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\OUTPUTS\nasa_2020_2025.csv
  [OK] mapping: C:\Users\USER\OneDrive\Escritorio\Github\DM_TF\BDS\mapping_cultivo_distrito.csv

Insumos listos.


## §2 Vista previa de insumos


In [8]:
df_midagri = pd.read_csv(insumos["midagri_largo.csv"])
df_nasa = pd.read_csv(insumos["nasa_2020_2025.csv"])
df_mapping = pd.read_csv(insumos["mapping"])

print(f"MIDAGRI:  {df_midagri.shape}")
print(f"NASA:     {df_nasa.shape}")
print(f"Mapping:  {df_mapping.shape}")
print(f"\nRegiones: {sorted(df_midagri['region'].unique())}")
df_midagri.head(3)


MIDAGRI:  (23328, 6)
NASA:     (1008, 19)
Mapping:  (200, 7)

Regiones: ['Ica', 'Junin', 'La Libertad', 'Piura', 'Puno', 'San Martin']


,region,anio,mes_num,mes_nombre,cultivo,produccion_mensual
0,Ica,2020,1,Enero,aceituna,0.0
1,Ica,2020,1,Enero,aji,635.0
2,Ica,2020,1,Enero,ajo,12.0


## §3 Integración y exportación

Lógica en [`SCRIPTS/pipeline_integrado.py`](SCRIPTS/pipeline_integrado.py).


In [9]:
from pipeline_integrado import (
    construir_datasets,
    validar,
    exportar,
    filtrar_pareto,
)

dfs = construir_datasets(df_midagri, df_nasa, df_mapping)
stats = validar(dfs)
_, _, reporte_pareto = filtrar_pareto(dfs["dataset_por_cultivo"])

exportar(dfs, RUTA_OUTPUT)

print("=== Pareto-80 (corregido) ===")
for line in reporte_pareto:
    print(line)

print("\n=== Validación ===")
for k, v in stats.items():
    print(f"  {k}: {v}")

df_integrado = dfs["dataset_integrado"]
print(f"\nDataset integrado: {df_integrado.shape}")
df_integrado.head()


=== Pareto-80 (corregido) ===
Ica          : 7 cultivos (81.3% acumulado)
Junin        : 8 cultivos (81.7% acumulado)
La Libertad  : 4 cultivos (82.1% acumulado)
Piura        : 4 cultivos (90.6% acumulado)
Puno         : 3 cultivos (92.0% acumulado)
San Martin   : 4 cultivos (84.4% acumulado)

=== Validación ===
  max_diff_a_b: 2.3283064365386963e-10
  nan_clima: 0
  nan_prod_b: 1186
  nan_prod_a: 70
  shapes: {'dataset_por_cultivo': (14400, 20), 'dataset_regional': (1008, 20), 'dataset_integrado': (2160, 20)}

Dataset integrado: (2160, 20)


,region,piso_asignado,distrito,cultivo,anio,mes_num,mes_nombre,produccion_mensual,t2m,t2m_max,t2m_min,prectotcorr,rh2m,allsky_sfc_sw_dwn,ws2m,ps,gwetroot,ts,t2mdew,qv2m
4,Ica,costa,Chincha Alta,alfalfa,2020,1,Enero,10684.89,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
11,Ica,costa,Chincha Alta,esparrago,2020,1,Enero,14799.60,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
12,Ica,costa,Chincha Alta,maiz_amarillo_duro,2020,1,Enero,9000.19,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
14,Ica,costa,Chincha Alta,mandarina,2020,1,Enero,0.00,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41
18,Ica,costa,Chincha Alta,palta,2020,1,Enero,7.10,21.96,27.55,18.21,0.12,78.45,22.25,2.99,95.82,0.33,23.95,17.83,13.41


## §4 Resumen del dataset maestro

`dataset_integrado.csv` = cultivos Pareto-80 × 72 meses × clima asignado por piso ecológico.

**Uso en clustering:**
```python
df = pd.read_csv("OUTPUTS/dataset_integrado.csv")
```


In [10]:
df_integrado = pd.read_csv(RUTA_OUTPUT / "dataset_integrado.csv")

print("=== dataset_integrado.csv ===")
print(f"Filas: {len(df_integrado):,}  |  Columnas: {len(df_integrado.columns)}")
print(f"Combinaciones (region, cultivo): {df_integrado.groupby(['region','cultivo']).ngroups}")
print(f"NaN en produccion_ton: {df_integrado['produccion_ton'].isna().sum()}")
print(f"\nColumnas:\n  {list(df_integrado.columns)}")
print("\n=== Archivos exportados ===")
for f in sorted(RUTA_OUTPUT.glob("dataset*.csv")):
    n = sum(1 for _ in open(f, encoding='utf-8-sig')) - 1
    print(f"  {f.name}: {n:,} filas")

df_integrado.describe().round(2)


=== dataset_integrado.csv ===
Filas: 2,160  |  Columnas: 20
Combinaciones (region, cultivo): 30


KeyError: 'produccion_ton'